In [ ]:
#Checking PyTorch and GPU
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
#Getting GPU Information
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

In [ ]:
#Installing Required Libraries

!pip install torch torchvision torchaudio transformers datasets accelerate

In [ ]:
from transformers import pipeline

In [ ]:
#Summarization

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

text = "Artificial intelligence is transforming the world."

inputs = tokenizer(text, return_tensors="pt")

summary_ids = model.generate(inputs["input_ids"], max_length=40)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary)

In [ ]:
!pip install transformers datasets torch gradio accelerate

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
dataset = load_dataset("cnn_dailymail", "3.0.0")

print(dataset)

In [ ]:
print(dataset["train"][0]["article"])
print(dataset["train"][0]["highlights"])

In [ ]:
model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
def preprocess(example):

    inputs = tokenizer(
        example["article"],
        max_length=512,
        truncation=True
    )

    targets = tokenizer(
        example["highlights"],
        max_length=128,
        truncation=True
    )

    inputs["labels"] = targets["input_ids"]

    return inputs

In [ ]:
tokenized_dataset = dataset["train"].select(range(2000)).map(preprocess)

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=50,
    save_steps=500,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

In [ ]:
!pip install evaluate
!pip install rouge_score


In [ ]:
import evaluate

rouge = evaluate.load("rouge")

results = rouge.compute(
    predictions=["generated summary"],
    references=["actual summary"]
)

print(results)

In [ ]:
def summarize(text):

    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True)

    # Move the input tensors to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=120,
        min_length=30
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    return summary

In [ ]:
text = "Based on its paper, GPT-3 is an autoregressive language model as opposed to a denoising autoencoder like BERT. I decided to write about some of the comparative differences between those two architectures of language models. The paper is an investigation into what you can do with giant language models. Now this language model is an order of magnitude larger than anyone has ever built a language model."

print(summarize(text))

In [ ]:
model.save_pretrained("./fine_tuned_bart_model")
tokenizer.save_pretrained("./fine_tuned_bart_model")
print("Model and tokenizer saved to ./fine_tuned_bart_model")